# 06 — Machine Learning Experiments & Honest Limitations

### Scope & Integrity Disclosures
1. **Experiment A — Chronological Forecasting of Karnataka Monthly EV Sales (VAHAN):**
   - Uses a **strict chronological split** (Train: `Apr 2021 – Jun 2023`, Test: `Jul 2023 – Mar 2024`) to prevent temporal leakage.
   - Compares a **Linear Regression baseline** against a **Random Forest Regressor** using MAE, RMSE, and $R^2$.
2. **Experiment B — Unsupervised Infrastructure Clustering (K-Means):**
   - Segments Karnataka charging stations by location, power kW, connector count, and confidence score.
   - **Important limitation:** Neither model predicts real-time Karnataka charger occupancy because historical Indian CPO live session logs are not publicly available.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

REPO = Path("..").resolve()
PROC = REPO / "data" / "processed"

# Experiment A: Chronological EV Sales Forecasting (Karnataka 2W + 4W monthly)
ka = pd.read_csv(PROC / "karnataka_ev_sales_clean.csv", parse_dates=["date_parsed"])
ka = ka.sort_values(["vehicle_category", "date_parsed"]).reset_index(drop=True)
ka["is_4w"] = (ka["vehicle_category"] == "4-Wheelers").astype(int)
ka["month_num"] = ka["date_parsed"].dt.month
ka["time_idx"] = (ka["date_parsed"].dt.year - 2021) * 12 + ka["date_parsed"].dt.month
ka["lag_1"] = ka.groupby("vehicle_category")["electric_vehicles_sold"].shift(1)
ka["lag_2"] = ka.groupby("vehicle_category")["electric_vehicles_sold"].shift(2)
ka["rolling_3"] = ka.groupby("vehicle_category")["electric_vehicles_sold"].transform(lambda s: s.shift(1).rolling(3).mean())
ka = ka.dropna().reset_index(drop=True)

split_date = pd.Timestamp("2023-07-01")
train = ka[ka["date_parsed"] < split_date]
test = ka[ka["date_parsed"] >= split_date]

features = ["time_idx", "month_num", "is_4w", "lag_1", "lag_2", "rolling_3"]
X_train, y_train = train[features], train["electric_vehicles_sold"]
X_test, y_test = test[features], test["electric_vehicles_sold"]

print(f"Chronological split -> Train rows: {len(train)} (up to {train['date_parsed'].max().date()}), Test rows: {len(test)} (from {test['date_parsed'].min().date()})")

lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(X_train, y_train)

results = []
for name, model in [("Linear Regression (Baseline)", lr), ("Random Forest Regressor", rf)]:
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "MAE": round(mean_absolute_error(y_test, preds), 2),
        "RMSE": round(root_mean_squared_error(y_test, preds), 2),
        "R2": round(r2_score(y_test, preds), 4),
    })

display(pd.DataFrame(results))

Chronological split -> Train rows: 48 (up to 2023-06-01), Test rows: 18 (from 2023-07-01)


,Model,MAE,RMSE,R2
0,Linear Regression (Baseline),1988.82,3329.11,0.7263
1,Random Forest Regressor,1322.57,2447.48,0.8521


In [2]:
# Experiment B: K-Means Spatial + Power Clustering of Karnataka Stations
stations = pd.read_csv(PROC / "karnataka_stations_clean.csv")
stations["power_imputed"] = stations["max_power_kw"].fillna(stations["max_power_kw"].median())
stations["power_missing_flag"] = stations["max_power_kw"].isna().astype(int)

cluster_feats = ["latitude", "longitude", "power_imputed", "total_connectors", "confidence_score", "power_missing_flag"]
X_scaled = StandardScaler().fit_transform(stations[cluster_feats])

sil_rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sil_rows.append({"k": k, "inertia": round(km.inertia_, 2), "silhouette": round(silhouette_score(X_scaled, km.labels_), 4)})

sil_df = pd.DataFrame(sil_rows)
display(sil_df)

best_k = int(sil_df.sort_values("silhouette", ascending=False).iloc[0]["k"])
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_scaled)
stations["cluster"] = km_best.labels_
print(f"\nBest K by Silhouette Score: k={best_k}")
display(stations.groupby("cluster").agg(
    stations=("station_id", "count"),
    mean_lat=("latitude", "mean"),
    mean_lon=("longitude", "mean"),
    median_power_kw=("max_power_kw", "median"),
    mean_confidence=("confidence_score", "mean"),
).round(3))

,k,inertia,silhouette
0,2,297.72,0.6047
1,3,182.29,0.6433
2,4,109.72,0.6761
3,5,89.46,0.6755
4,6,72.54,0.6918



Best K by Silhouette Score: k=6


,stations,mean_lat,mean_lon,median_power_kw,mean_confidence
cluster,,,,,
0,28,13.019,77.528,NaN,0.450
1,8,12.698,75.864,60.0,0.900
2,13,16.485,75.403,NaN,0.542
3,7,12.632,76.313,22.0,0.796
4,28,12.976,77.610,NaN,0.620
5,2,12.531,75.933,105.0,0.920
